# Delta Lake with PySpark — comprehensive examples

Based on **001-DeltaTable (1).ipynb**, for **Delta 3.3.2**, **Spark 3.5.x**, **Scala 2.12**,
**HDFS**, and **Hive Metastore**.

Use the fixed database **`orderdb`**, table names such as **`orderdb.orders`**, and explicit
HDFS locations. There are no generated names, runtime profiles, table factories, or SQL
placeholder dictionaries. Examples use `spark.sql(...)`, `%%sql`, and the DeltaTable API.

Edit the Spark master and service hostnames in section 3. Run the lessons in order on empty
training tables. Restarting a kernel does not reset tables: inspect existing tables before
using the commented cleanup commands. External registrations and their HDFS data are separate.

Version variables save actual Delta commit versions for time travel, RESTORE, and CDF.
The advanced coverage remains: CRUD, MERGE, SCD2, schemas, constraints, partitioning, clustering,
deletion vectors, row tracking, identity columns, streaming, conversion, and maintenance.
Optional streaming, negative-error examples, and cleanup are marked in their cells.

## 1. Check Java and Spark

Use a Python Jupyter kernel on your Spark host. The cluster needs Spark 3.5.x / Scala 2.12
and the Delta 3.3.2 JARs. Python also needs the `delta-spark` package.

In [ ]:
!java -version
# !spark-submit --version
# !hdfs dfs -ls /user/hive/warehouse

## 2. Install Delta once on the Spark host

The commented Bash commands below are your two JAR downloads. Uncomment and run once on the
Linux Spark host if the JARs are not installed, then restart Spark and the notebook kernel.
Use the same versions on the driver and workers. `-k` retains your TLS setting; omit it when
the host's trusted certificates work. Install the Python package using the next section.

In [ ]:
%%bash
# curl -k --fail --location --retry 5 --continue-at - \
#   --output "$SPARK_HOME/jars/delta-spark_2.12-3.3.2.jar" \
#   https://repo.maven.apache.org/maven2/io/delta/delta-spark_2.12/3.3.2/delta-spark_2.12-3.3.2.jar

# curl -k --fail --location --retry 5 --continue-at - \
#   --output "$SPARK_HOME/jars/delta-storage-3.3.2.jar" \
#   https://repo.maven.apache.org/maven2/io/delta/delta-storage/3.3.2/delta-storage-3.3.2.jar

**If PySpark 3.5.x is already available in this kernel**, install only the Delta Python wrapper:

```python
%pip install --no-deps delta-spark==3.3.2
```

For a new kernel environment, install the matching PySpark patch and Delta package together.
The following matches the Spark patch recorded in your earlier Iceberg notebook; substitute
your actual cluster patch if different:

```python
%pip install pyspark==3.5.9 delta-spark==3.3.2
```

Restart the kernel after installation. Keep the Python PySpark version aligned with your cluster;
do not let an unrelated Spark 4 installation supply the kernel or executor libraries.

Example Linux environment before `jupyter lab`:

```bash
export HADOOP_CONF_DIR=/path/to/hadoop/conf
jupyter lab
```

Use `core-site.xml`, `hdfs-site.xml`, and Kerberos credentials as required by your cluster.
Edit the Spark master, Hive Metastore URI, and HDFS warehouse directly in section 3.
`localhost` works only when the referenced service is local to the process using that address.

## 3. Configure Spark for OSS Delta, Hive Metastore and HDFS

This is the main setup cell. `spark_catalog` is wrapped by **DeltaCatalog**, while Hive support
persists table registrations in Hive Metastore. The log in HDFS remains the transactional authority.
Use `spark.hadoop.hive.metastore.uris` rather than the non-Spark builder key that produced a
warning in the earlier notebook. A Hive registration alone does not make plain Hive/Parquet
readers Delta-aware. [Quick start](https://docs.delta.io/quick-start/),
[storage](https://docs.delta.io/delta-storage/).

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from delta.tables import DeltaTable
from decimal import Decimal

spark = (
    SparkSession.builder
    .appName("Delta-PySpark-Examples")
    .master("spark://your-spark-host:7077")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9083")
    .config("spark.sql.warehouse.dir", "hdfs://localhost:9000/user/hive/warehouse")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.databricks.delta.retentionDurationCheck.enabled", "true")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

## 4. Enable simple Spark SQL cells

This magic passes the cell directly to `spark.sql`. Use one statement per `%%sql` cell.
Use a Python f-string only when inserting a captured commit version or timestamp.

In [ ]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def sql(line, cell):
    spark.sql(cell).show(50, truncate=False)

## 5. Create the orderdb database

Use `orderdb` throughout. The main examples are external Delta tables with explicit HDFS
locations. The managed-table example has its own subdirectory.

In [ ]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS orderdb
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/managed'
""")
spark.sql("SHOW DATABASES").show()
spark.sql("SHOW TABLES IN orderdb").show()

## 6. Original orders schema, with exact decimal amounts

The source used `FLOAT` for money; this version uses `DECIMAL(12,2)` for reproducible comparisons.
CDF is enabled before the first insert. Deletion vectors are disabled on this table so the
original copy-on-write lesson remains visible; a dedicated DV lab appears later.

In [ ]:
spark.sql("""CREATE TABLE orderdb.orders (
    order_id INT NOT NULL, customer_id INT, product_id INT, amount DECIMAL(12,2)
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/orders'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false',
    'delta.enableChangeDataFeed'='true',
    'delta.logRetentionDuration'='interval 30 days',
    'delta.deletedFileRetentionDuration'='interval 7 days'
)""")
empty_version = DeltaTable.forName(spark, 'orderdb.orders').history(1).first().version
spark.sql('DESCRIBE EXTENDED orderdb.orders').show(truncate=False)
spark.sql('DESCRIBE DETAIL orderdb.orders').show(truncate=False)
spark.sql('SHOW TBLPROPERTIES orderdb.orders').show(truncate=False)

## 7. INSERT and capture the version/timestamp

Expected: one order, ID 1, amount 123.40. Capture the actual committed version instead of
assuming the insert is always version 1. Epoch milliseconds are computed in Spark, avoiding
driver OS timezone conversion errors.

In [ ]:
%%sql
INSERT INTO orderdb.orders VALUES (1, 10, 100, CAST(123.40 AS DECIMAL(12,2)))

In [ ]:
insert_version = DeltaTable.forName(spark, "orderdb.orders").history(1).first().version
insert_timestamp = (
    DeltaTable.forName(spark, "orderdb.orders").history(1)
    .selectExpr("CAST(timestamp AS STRING)").first()[0]
)
print("Insert version:", insert_version)
print("Insert timestamp:", insert_timestamp)
spark.sql("SELECT * FROM orderdb.orders").show()

## 8. UPDATE and DELETE with immutable files

Expected sequence: amount becomes 150.00, then the current table is empty. Logical removal
does not immediately erase old files. Delta readers consult `_delta_log`; reading the directory
as raw Parquet can expose stale or duplicate row images and bypass Delta features.

In [ ]:
spark.sql('UPDATE orderdb.orders SET amount=CAST(150 AS DECIMAL(12,2)) WHERE order_id=1').show(truncate=False)
spark.sql('SELECT * FROM orderdb.orders').show()
update_version = DeltaTable.forName(spark, 'orderdb.orders').history(1).first().version
spark.sql('DELETE FROM orderdb.orders WHERE order_id=1').show(truncate=False)
delete_version = DeltaTable.forName(spark, 'orderdb.orders').history(1).first().version
spark.sql('DESCRIBE HISTORY orderdb.orders').show(truncate=False)
spark.sql('SELECT * FROM orderdb.orders').show()

## 9. Inspect the transaction log and history

Numbered JSON commits contain actions such as `add`, `remove`, `metaData`, `protocol`, and
`commitInfo`. A remove action marks a file inactive; it is not an instruction to delete it manually.
Checkpoints summarize state so readers do not replay the entire history. This is a read-only
teaching inspection, not a replacement for the Delta reader or a production log parser.

In [ ]:
# The JSON filenames below are the first four commits on a new orders table.
# Inspect DESCRIBE HISTORY if reusing a table from another session.
spark.read.text(
    "hdfs://localhost:9000/user/hive/warehouse/orderdb/orders/_delta_log/00000000000000000000.json"
).show(truncate=False)
spark.read.text(
    "hdfs://localhost:9000/user/hive/warehouse/orderdb/orders/_delta_log/00000000000000000001.json"
).show(truncate=False)
spark.read.text(
    "hdfs://localhost:9000/user/hive/warehouse/orderdb/orders/_delta_log/00000000000000000002.json"
).show(truncate=False)
spark.read.text(
    "hdfs://localhost:9000/user/hive/warehouse/orderdb/orders/_delta_log/00000000000000000003.json"
).show(truncate=False)
DeltaTable.forName(spark, "orderdb.orders").history().show(truncate=False)

## 10. Time travel by version and timestamp

These reads leave the current empty table unchanged. The initial insert version has 123.40;
the update version has 150.00; the create version is empty. This corrects the attachment's
example that selected version 0 while describing a populated version. Historical queries need
both retained log state and the referenced data files.

In [ ]:
spark.sql(f"""
SELECT * FROM orderdb.orders VERSION AS OF {insert_version}
""").show()

In [ ]:
spark.sql(f"""
SELECT * FROM orderdb.orders TIMESTAMP AS OF '{insert_timestamp}'
""").show()

In [ ]:
historical = (
    spark
    .read
    .format('delta')
    .option('versionAsOf', insert_version)
    .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
)
by_time = (
    spark
    .read
    .format('delta')
    .option('timestampAsOf', insert_timestamp)
    .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
)
historical.show()
by_time.show()

## 11. RESTORE by version and timestamp

Delta RESTORE **creates a new commit** whose contents match the selected historical state;
it does not move a branch pointer back to an old version. Restoring the saved empty state is
also possible while its files/log remain available. RESTORE is data-changing for downstream
consumers and can reintroduce rows to streams. [RESTORE reference](https://docs.delta.io/delta-utility/#restore-a-delta-table-to-an-earlier-state).

In [ ]:
spark.sql(f'RESTORE TABLE orderdb.orders TO VERSION AS OF {update_version}').show(truncate=False)
spark.sql('SELECT * FROM orderdb.orders').show()
restore_version = DeltaTable.forName(spark, 'orderdb.orders').history(1).first().version
spark.sql(f"RESTORE TABLE orderdb.orders TO TIMESTAMP AS OF '{insert_timestamp}'").show(truncate=False)
spark.sql('SELECT * FROM orderdb.orders').show()
DeltaTable.forName(spark, 'orderdb.orders').restoreToVersion(delete_version).show(truncate=False)
spark.sql('SELECT * FROM orderdb.orders').show()
DeltaTable.forName(spark, 'orderdb.orders').restoreToVersion(update_version).show(truncate=False)
spark.sql('SELECT * FROM orderdb.orders').show()
spark.sql('DESCRIBE HISTORY orderdb.orders').show(truncate=False)

## 12. Snapshot differences and consistent named/path reads

Use `exceptAll` to compare row images with multiplicity. An update appears as one removed
and one added image. For reproducible multi-query analysis, pin the same version in every
read. Copies made later may preserve version numbers but change filesystem timestamps;
prefer version anchors for portability.

In [ ]:
old = (
    spark
    .read
    .format('delta')
    .option('versionAsOf', insert_version)
    .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
)
new = (
    spark
    .read
    .format('delta')
    .option('versionAsOf', update_version)
    .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
)
print('Removed images:')
old.exceptAll(new).show()
print('Added images:')
new.exceptAll(old).show()

## 13. Deterministic MERGE with inserts, updates and deletes

Extend the original small-file sample with a CDC-style batch. Deduplicate each source key
before MERGE using a unique sequence/event ordering. An update assignment can be replayable,
but arbitrary increments are not. Here the highest sequence wins for order 1; order 4 is
deleted; order 8 is inserted. Expected IDs: **1, 3, 6, 7, 8**.

In [ ]:
(
    spark
    .sql('INSERT INTO orderdb.orders VALUES (7,7,7,7.00),(6,6,6,6.00),(4,4,4,4.00),(3,3,3,3.00)')
    .show(truncate=False)
)
seeded_version = DeltaTable.forName(spark, 'orderdb.orders').history(1).first().version
from pyspark.sql.window import Window
incoming = spark.createDataFrame([(1, 10, 100, Decimal('190.00'), 1, 'evt1', 'U'), (1, 10, 100, Decimal('200.00'), 2, 'evt2', 'U'), (4, 4, 4, Decimal('4.00'), 1, 'evt3', 'D'), (8, 80, 800, Decimal('80.00'), 1, 'evt4', 'I')], 'order_id int, customer_id int, product_id int, amount decimal(12,2), seq long, event_id string, op string')
latest = (
    incoming
    .withColumn('rn', F.row_number().over(Window.partitionBy('order_id').orderBy(F.desc('seq'), F.desc('event_id'))))
    .where('rn=1')
    .drop('rn')
)
latest.createOrReplaceTempView('order_changes')
spark.sql("""MERGE INTO orderdb.orders t USING order_changes s ON t.order_id=s.order_id
WHEN MATCHED AND s.op='D' THEN DELETE
WHEN MATCHED AND s.op<>'D' THEN UPDATE SET
  t.customer_id=s.customer_id, t.product_id=s.product_id, t.amount=s.amount
WHEN NOT MATCHED AND s.op<>'D' THEN INSERT (order_id, customer_id, product_id, amount)
  VALUES (s.order_id, s.customer_id, s.product_id, s.amount)""").show(truncate=False)
merged_version = DeltaTable.forName(spark, 'orderdb.orders').history(1).first().version
spark.sql('SELECT * FROM orderdb.orders ORDER BY order_id').show(truncate=False)

## 14. Python DeltaTable API and full-source synchronization

This independent table demonstrates Python `update`, `delete`, and `merge`, including
`whenNotMatchedBySourceDelete`. Use that last clause only when the source is a complete,
appropriately scoped snapshot; a partial CDC batch must not delete all absent target keys.
[DML reference](https://docs.delta.io/delta-update/).

In [ ]:
spark.sql("""CREATE TABLE orderdb.python_api (
    id INT, label STRING
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/python_api'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
spark.sql("INSERT INTO orderdb.python_api VALUES (1,'old'),(2,'retire'),(3,'temporary')").show(truncate=False)
api = DeltaTable.forName(spark, 'orderdb.python_api')
api.update(condition='id=1', set={'label': F.lit('updated')})
api.delete('id=3')
complete_source = spark.createDataFrame([(1, 'current'), (4, 'new')], 'id int, label string')
(
    api
    .alias('t')
    .merge(complete_source.alias('s'), 't.id=s.id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)
api.toDF().show()

## 15. Change Data Feed: bounded batch reads

CDF records logical changes after it is enabled. Version bounds are **inclusive**: start at
`seeded_version + 1` to exclude the seeded snapshot. Inspect insert/delete and update before/after
images. CDF is retained with the table's history, not an indefinite audit archive.
Non-additive schema changes and column mapping require special care when selecting ranges.
[CDF reference](https://docs.delta.io/delta-change-data-feed/).

In [ ]:
cdf = (
    spark
    .read
    .format('delta')
    .option('readChangeFeed', 'true')
    .option('startingVersion', seeded_version + 1)
    .option('endingVersion', merged_version)
    .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
)
cdf.orderBy('_commit_version', 'order_id', '_change_type').show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT order_id, amount, _change_type, _commit_version, _commit_timestamp
FROM table_changes('orderdb.orders', {seeded_version + 1}, {merged_version})
ORDER BY _commit_version, order_id, _change_type
""").show(truncate=False)

## 16. Replay-safe append transactions

`txnAppId` and `txnVersion` deduplicate a repeated write to one Delta table. Reusing a pair
with different data will not apply the changed payload. Use increasing transaction versions
per application, and a new application ID when intentionally restarting with reset progress.
This mechanism does not make writes to several tables atomic.

In [ ]:
spark.sql("""CREATE TABLE orderdb.idempotent_append (
    id LONG, payload STRING
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/idempotent_append'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
batch = spark.createDataFrame([(1, 'a'), (2, 'b')], 'id long, payload string')

def append_batch(frame, batch_id):
    (
        frame
        .write
        .format('delta')
        .mode('append')
        .option('txnAppId', 'orders_ingestion')
        .option('txnVersion', int(batch_id))
        .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/idempotent_append')
    )
append_batch(batch, 0)
append_batch(batch, 0)
append_batch(spark.createDataFrame([(3, 'c')], batch.schema), 1)
spark.sql('SELECT * FROM orderdb.idempotent_append ORDER BY id').show()

## 17. SCD Type 2 with one atomic MERGE

Keep effective-date ranges as `[effective_from, effective_to)` and exactly one current row per
customer. The staged source uses a null merge key for the new version of an existing customer,
plus a keyed row to close the previous version. Replayed identical changes stage nothing new.
This small example assumes one ordered effective change per key per batch and no concurrent
writer; late-arriving historical corrections require additional interval logic.

In [ ]:
spark.sql("""CREATE TABLE orderdb.customer_scd2 (
    customer_id INT, name STRING, city STRING, effective_from DATE, effective_to DATE, is_current BOOLEAN
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/customer_scd2'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
(
    spark
    .sql("INSERT INTO orderdb.customer_scd2 VALUES (1,'Alice','Bangalore',DATE '2026-01-01',NULL,true),(2,'Bob','Hyderabad',DATE '2026-01-01',NULL,true)")
    .show(truncate=False)
)
scd_source = spark.sql("""SELECT 1 AS customer_id, 'Alice' AS name, 'Pune' AS city, DATE '2026-02-01' AS effective_from
UNION ALL SELECT 3, 'Carol', 'Chennai', DATE '2026-02-01'""")

def apply_scd2(source):
    current = spark.table('orderdb.customer_scd2').where('is_current').alias('t')
    changed = (
        source
        .alias('s')
        .join(current, F.col('s.customer_id') == F.col('t.customer_id'))
        .where('NOT (s.name <=> t.name AND s.city <=> t.city)')
        .select('s.*')
    )
    staged_rows = (
        source
        .withColumn('merge_key', F.col('customer_id'))
        .unionByName(changed.withColumn('merge_key', F.lit(None).cast('int')))
        .collect()
    )
    staged_schema = T.StructType(list(source.schema.fields) + [T.StructField('merge_key', T.IntegerType(), True)])
    staged = spark.createDataFrame(staged_rows, staged_schema)
    (
        DeltaTable
        .forName(spark, 'orderdb.customer_scd2')
        .alias('t')
        .merge(staged.alias('s'), 't.customer_id=s.merge_key AND t.is_current=true')
        .whenMatchedUpdate(condition='NOT (s.name <=> t.name AND s.city <=> t.city)', set={'is_current': 'false', 'effective_to': 's.effective_from'})
        .whenNotMatchedInsert(values={'customer_id': 's.customer_id', 'name': 's.name', 'city': 's.city', 'effective_from': 's.effective_from', 'effective_to': 'CAST(NULL AS DATE)', 'is_current': 'true'})
        .execute()
    )
apply_scd2(scd_source)
apply_scd2(scd_source)
spark.sql('SELECT * FROM orderdb.customer_scd2 ORDER BY customer_id, effective_from').show(truncate=False)

## 18. Schema enforcement and explicit additive evolution

Unexpected source fields should be reviewed. The commented append shows a schema mismatch if you run it; the main append
uses per-write `mergeSchema`. Existing rows receive null for the new column. Prefer a scoped
option over enabling schema auto-merge globally for every table in the session.

In [ ]:
spark.sql("""CREATE TABLE orderdb.schema_evolution (
    id INT, profile STRUCT<city:STRING>
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/schema_evolution'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
spark.sql("INSERT INTO orderdb.schema_evolution VALUES (1,named_struct('city','Pune'))").show(truncate=False)
new_schema = spark.sql("SELECT 2 AS id, named_struct('city','Delhi') AS profile, 'crm' AS source_system")
(
    new_schema
    .write
    .format('delta')
    .mode('append')
    .option('mergeSchema', 'true')
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/schema_evolution')
)
(
    spark
    .sql('ALTER TABLE orderdb.schema_evolution ADD COLUMNS (profile.country STRING, email STRING)')
    .show(truncate=False)
)
(
    spark
    .sql("ALTER TABLE orderdb.schema_evolution ALTER COLUMN email COMMENT 'Contact email when supplied'")
    .show(truncate=False)
)
spark.sql('SELECT * FROM orderdb.schema_evolution').show(truncate=False)

# Optional negative example: uncomment to see the expected error.
# new_schema.write.format('delta').mode('append').save('hdfs://localhost:9000/user/hive/warehouse/orderdb/schema_evolution')

## 18a. MERGE with per-operation schema evolution

The DeltaMergeBuilder in 3.3.2 supports `withSchemaEvolution`. This is a **MERGE builder** method;
do not assume newer DataFrameWriter methods from Spark/Delta 4.x exist in this runtime.
The new source column is added to the target only for this explicit operation.

In [ ]:
spark.sql("""CREATE TABLE orderdb.merge_schema_evolution (
    id INT, label STRING
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/merge_schema_evolution'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
spark.sql("INSERT INTO orderdb.merge_schema_evolution VALUES (1,'old')").show(truncate=False)
evolving_source = spark.createDataFrame([(1, 'updated', 'crm'), (2, 'new', 'web')], 'id int, label string, source_system string')
(
    DeltaTable
    .forName(spark, 'orderdb.merge_schema_evolution')
    .alias('t')
    .merge(evolving_source.alias('s'), 't.id=s.id')
    .withSchemaEvolution()
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
spark.sql('SELECT * FROM orderdb.merge_schema_evolution').show(truncate=False)

## 19. Column mapping: rename and drop without rewriting files

Name-based column mapping separates logical names from physical Parquet names. A rename preserves
the field; a drop removes it from the logical schema but does not erase historical file bytes.
This table enables the required protocol explicitly. Inspect reader/writer features before
sharing it with older engines. [Column mapping](https://docs.delta.io/delta-column-mapping/).

In [ ]:
spark.sql("""CREATE TABLE orderdb.column_mapping (
    id INT, old_name STRING, obsolete STRING
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/column_mapping'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false',
    'delta.columnMapping.mode'='name',
    'delta.minReaderVersion'='2',
    'delta.minWriterVersion'='5'
)""")
spark.sql("INSERT INTO orderdb.column_mapping VALUES (1,'Alice','old payload')").show(truncate=False)
spark.sql('ALTER TABLE orderdb.column_mapping RENAME COLUMN old_name TO customer_name').show(truncate=False)
spark.sql('ALTER TABLE orderdb.column_mapping DROP COLUMN obsolete').show(truncate=False)
spark.sql('DESCRIBE DETAIL orderdb.column_mapping').show(truncate=False)

## 20. Full schema replacement on a disposable table

`overwriteSchema` is an explicit replacement, not ordinary additive evolution. The input comes
from the immutable orders version captured earlier, avoiding a lazy read of the target being
overwritten. This lab changes decimal money to a string only to demonstrate a schema replacement;
keep decimal types for real financial calculations.

In [ ]:
spark.sql("""CREATE TABLE orderdb.schema_replacement (
    order_id INT, amount DECIMAL(12,2)
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/schema_replacement'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
(
    spark
    .read
    .format('delta')
    .option('versionAsOf', merged_version)
    .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
    .select('order_id', 'amount')
    .write
    .format('delta')
    .mode('append')
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/schema_replacement')
)
(
    spark
    .read
    .format('delta')
    .option('versionAsOf', merged_version)
    .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
    .select('order_id', F.col('amount').cast('string').alias('amount_text'))
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/schema_replacement')
)
spark.catalog.refreshTable('orderdb.schema_replacement')

## 21. NOT NULL and CHECK constraints

Constraints reject an invalid write atomically. Adding a CHECK also validates existing data.
They do not provide a primary key, unique index, or foreign-key enforcement. The commented negative example shows a constraint error if you choose to run it.
[Constraint reference](https://docs.delta.io/delta-constraints/).

In [ ]:
spark.sql("""CREATE TABLE orderdb.constraints_lab (
    id INT NOT NULL, amount DECIMAL(12,2) NOT NULL
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/constraints_lab'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
(
    spark
    .sql('ALTER TABLE orderdb.constraints_lab ADD CONSTRAINT nonnegative_amount CHECK (amount >= 0)')
    .show(truncate=False)
)
spark.sql('INSERT INTO orderdb.constraints_lab VALUES (1,10.00)').show(truncate=False)
spark.sql('SHOW TBLPROPERTIES orderdb.constraints_lab').show(truncate=False)

# Optional negative example: uncomment to see the expected error.
# spark.sql('INSERT INTO orderdb.constraints_lab VALUES (2,-5.00)').collect()

## 22. Generated date columns and default values

A generated column derives and stores a value from other fields. A default supplies a value
when an insert omits the field or uses `DEFAULT`; it is not a generated expression tied to
another field. These examples use separate tables because feature requirements differ.
[Generated columns](https://docs.delta.io/delta-batch/#use-generated-columns),
[default columns](https://docs.delta.io/delta-default-columns/).

In [ ]:
(
    DeltaTable
    .create(spark)
    .tableName('orderdb.generated_dates')
    .location('hdfs://localhost:9000/user/hive/warehouse/orderdb/generated_dates')
    .addColumn('id', 'INT')
    .addColumn('event_ts', 'TIMESTAMP')
    .addColumn('event_date', 'DATE', generatedAlwaysAs='CAST(event_ts AS DATE)')
    .partitionedBy('event_date')
    .execute()
)
(
    spark
    .sql("INSERT INTO orderdb.generated_dates (id,event_ts) VALUES (1,TIMESTAMP '2026-01-01 12:00:00')")
    .show(truncate=False)
)
spark.sql('SELECT * FROM orderdb.generated_dates').show(truncate=False)
spark.sql("""CREATE TABLE orderdb.default_values (
    id INT, status STRING DEFAULT 'new'
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/default_values'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false',
    'delta.feature.allowColumnDefaults'='supported'
)""")
spark.sql('INSERT INTO orderdb.default_values (id) VALUES (1)').show(truncate=False)
spark.sql('INSERT INTO orderdb.default_values VALUES (2,DEFAULT)').show(truncate=False)
spark.sql("ALTER TABLE orderdb.default_values ALTER COLUMN status SET DEFAULT 'pending'").show(truncate=False)
spark.sql('INSERT INTO orderdb.default_values (id) VALUES (3)').show(truncate=False)

## 23. Partitioned tables and selective overwrite with replaceWhere

Choose a moderate-cardinality partition column. Here a replacement batch covers only one date;
the other date must survive. `replaceWhere` is an explicit predicate with input validation enabled.
Do not use full-table overwrite for a partial daily correction.

In [ ]:
spark.sql("""CREATE TABLE orderdb.daily_orders (
    id LONG, order_date DATE, region STRING, amount DECIMAL(12,2)
)
USING DELTA
PARTITIONED BY (order_date)
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/daily_orders'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
spark.sql("""INSERT INTO orderdb.daily_orders VALUES
 (1,DATE '2026-01-01','south',10.00), (2,DATE '2026-01-02','north',20.00)""").show(truncate=False)
day_fix = spark.sql("SELECT 3L AS id, DATE '2026-01-01' AS order_date, 'west' AS region, CAST(30 AS DECIMAL(12,2)) AS amount")
(
    day_fix
    .write
    .format('delta')
    .mode('overwrite')
    .option('replaceWhere', "order_date = '2026-01-01'")
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/daily_orders')
)
spark.sql('SELECT * FROM orderdb.daily_orders').show(truncate=False)
spark.sql("SELECT id,amount FROM orderdb.daily_orders WHERE order_date=DATE '2026-01-01'").explain('formatted')

## 24. Dynamic partition overwrite and layout changes

Dynamic overwrite replaces every partition represented by the incoming data. One incorrectly
dated record can therefore replace an unintended whole partition. Do not combine this writer
option with `replaceWhere`. Delta does not offer Iceberg-style hidden partition-spec evolution;
changing partition columns requires a planned rewrite or a new table.

In [ ]:
next_day = spark.sql("SELECT 4L AS id, DATE '2026-01-02' AS order_date, 'east' AS region, CAST(40 AS DECIMAL(12,2)) AS amount")
(
    next_day
    .write
    .format('delta')
    .mode('overwrite')
    .option('partitionOverwriteMode', 'dynamic')
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/daily_orders')
)
(
    spark
    .table('orderdb.daily_orders')
    .write
    .format('delta')
    .mode('errorifexists')
    .partitionBy('region')
    .option('path', 'hdfs://localhost:9000/user/hive/warehouse/orderdb/orders_by_region')
    .saveAsTable('orderdb.orders_by_region')
)
spark.sql('DESCRIBE DETAIL orderdb.orders_by_region').show(truncate=False)

## 25. OPTIMIZE: bin-packing and measurable file counts

OPTIMIZE is available in OSS Delta; it is not exclusively a Databricks command. The lab writes
several small batches, compacts them, and displays the active file counts. It does not
claim a speedup from a tiny teaching dataset. `numFiles` in DESCRIBE DETAIL counts active files.
[OSS optimizations](https://docs.delta.io/optimizations-oss/).

In [ ]:
spark.sql("""CREATE TABLE orderdb.layout_lab (
    id LONG, customer_id LONG, region STRING, amount LONG
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/layout_lab'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
for start in [0, 100, 200]:
    chunk = (
        spark
        .range(start, start + 100)
        .selectExpr('id', 'id % 17 AS customer_id', "CASE WHEN id % 2=0 THEN 'south' ELSE 'north' END AS region", 'id * 10 AS amount')
    )
    (
        chunk
        .repartition(4)
        .write
        .format('delta')
        .mode('append')
        .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/layout_lab')
    )
files_before = DeltaTable.forName(spark, 'orderdb.layout_lab').detail().first().numFiles
spark.sql('OPTIMIZE orderdb.layout_lab').show(truncate=False)
files_after = DeltaTable.forName(spark, 'orderdb.layout_lab').detail().first().numFiles
print('Active files before/after:', files_before, files_after)

## 26. Z-order, statistics and auto compaction

Z-order co-locates values to help selective reads skip files; it does not guarantee join or
GROUP BY acceleration. Pick columns used in useful predicates and with collected statistics.
Some OSS configuration names retain `spark.databricks.delta.*`; that prefix does not itself
mean a Databricks account is required.

In [ ]:
spark.sql('OPTIMIZE orderdb.layout_lab ZORDER BY (customer_id)').show(truncate=False)
spark.sql('SELECT * FROM orderdb.layout_lab WHERE customer_id=5').explain('formatted')
(
    spark
    .sql("ALTER TABLE orderdb.layout_lab SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true', 'delta.autoOptimize.autoCompact'='true')")
    .show(truncate=False)
)
DeltaTable.forName(spark, 'orderdb.layout_lab').optimize().executeCompaction().show(truncate=False)
spark.sql("OPTIMIZE orderdb.daily_orders WHERE order_date >= DATE '2026-01-01'").show(truncate=False)
spark.sql('DESCRIBE DETAIL orderdb.layout_lab').show(truncate=False)

## 27. Liquid clustering and OPTIMIZE FULL

Liquid clustering is also available in this OSS version. Use it on an unpartitioned table;
do not combine it with partitioning or Z-order. Changing clustering keys changes future layout
work; `OPTIMIZE ... FULL` in Delta 3.3 can recluster existing data. The small dataset may have
too few files to show a material layout difference. [Clustering reference](https://docs.delta.io/delta-clustering/).

In [ ]:
spark.sql("""CREATE TABLE orderdb.liquid_clustering (
    id LONG, customer_id LONG, region STRING, amount LONG
)
USING DELTA
CLUSTER BY (customer_id)
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/liquid_clustering'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
(
    spark
    .table('orderdb.layout_lab')
    .write
    .format('delta')
    .mode('append')
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/liquid_clustering')
)
spark.sql('OPTIMIZE orderdb.liquid_clustering').show(truncate=False)
spark.sql('ALTER TABLE orderdb.liquid_clustering CLUSTER BY (region, customer_id)').show(truncate=False)
spark.sql('OPTIMIZE orderdb.liquid_clustering FULL').show(truncate=False)
spark.sql('DESCRIBE DETAIL orderdb.liquid_clustering').show(truncate=False)

## 28. Deletion vectors and REORG

Deletion vectors mark removed row positions without always rewriting the original Parquet file.
The reader applies these marks to the current snapshot. Enable them on a separate table and
inspect the write actions. `REORG ... APPLY (PURGE)` materializes soft deletes in rewritten
current files; old files still need normal retention and VACUUM before physical deletion.
[Deletion vector reference](https://docs.delta.io/delta-deletion-vectors/).

In [ ]:
spark.sql("""CREATE TABLE orderdb.deletion_vectors (
    id LONG, value LONG
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/deletion_vectors'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='true'
)""")
(
    spark
    .range(100)
    .selectExpr('id', 'id * 10 AS value')
    .coalesce(1)
    .write
    .format('delta')
    .mode('append')
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/deletion_vectors')
)
spark.sql('DELETE FROM orderdb.deletion_vectors WHERE id=2').show(truncate=False)
dv_delete_version = DeltaTable.forName(spark, 'orderdb.deletion_vectors').history(1).first().version
dv_commit = (
    spark
    .read
    .text(f'hdfs://localhost:9000/user/hive/warehouse/orderdb/deletion_vectors/_delta_log/{dv_delete_version:020d}.json')
)
(
    dv_commit
    .select(F.get_json_object('value', '$.add.path').alias('file'), F.get_json_object('value', '$.add.deletionVector').alias('deletion_vector'))
    .show(truncate=False)
)
spark.sql('UPDATE orderdb.deletion_vectors SET value=999 WHERE id=1').show(truncate=False)
spark.sql('REORG TABLE orderdb.deletion_vectors APPLY (PURGE)').show(truncate=False)
spark.sql('DESCRIBE DETAIL orderdb.deletion_vectors').show(truncate=False)

## 29. Row tracking and row commit versions

Row IDs identify physical row lineage within a Delta table; they are not business keys or CDF
event IDs. After an UPDATE, the row ID remains while the row commit version changes. Use the
metadata fields explicitly. [Row tracking](https://docs.delta.io/delta-row-tracking/).

In [ ]:
spark.sql("""CREATE TABLE orderdb.row_tracking (
    id INT, label STRING
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/row_tracking'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false',
    'delta.enableRowTracking'='true'
)""")
spark.sql("INSERT INTO orderdb.row_tracking VALUES (1,'before'),(2,'unchanged')").show(truncate=False)
tracked_before = (
    spark
    .sql('SELECT id, _metadata.row_id AS row_id, _metadata.row_commit_version AS commit_version FROM orderdb.row_tracking')
    .where('id=1')
    .first()
)
spark.sql("UPDATE orderdb.row_tracking SET label='after' WHERE id=1").show(truncate=False)
tracked_after = (
    spark
    .sql('SELECT id, _metadata.row_id AS row_id, _metadata.row_commit_version AS commit_version FROM orderdb.row_tracking')
    .where('id=1')
    .first()
)
(
    spark
    .sql('SELECT *, _metadata.row_id, _metadata.row_commit_version FROM orderdb.row_tracking')
    .show(truncate=False)
)
print("Row tracking before:", tracked_before)
print("Row tracking after:", tracked_after)

## 30. Type widening: a narrow, version-compatible example

Delta 3.x type widening has a more limited type matrix than Delta 4.x. This example uses
SMALLINT to INT, not an assumed arbitrary cast. Turning the property off prevents further
widening but does not undo the protocol feature or previous changes. Test all external readers
before enabling it on a shared table. [Type widening matrix](https://docs.delta.io/delta-type-widening/).

In [ ]:
spark.sql("""CREATE TABLE orderdb.type_widening (
    id INT, small_value SMALLINT
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/type_widening'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false',
    'delta.enableTypeWidening'='true'
)""")
spark.sql('INSERT INTO orderdb.type_widening VALUES (1,CAST(100 AS SMALLINT))').show(truncate=False)
spark.sql('ALTER TABLE orderdb.type_widening ALTER COLUMN small_value TYPE INT').show(truncate=False)
spark.sql('INSERT INTO orderdb.type_widening VALUES (2,50000)').show(truncate=False)
spark.sql('DESCRIBE DETAIL orderdb.type_widening').show(truncate=False)

## 31. Protocol versions, append-only tables and feature lifecycle

The Delta **library version**, the table's **commit version**, and its **reader/writer protocol**
are different numbers. Features can raise protocol requirements. Setting a feature property to
false need not remove its protocol feature; supported `DROP FEATURE` workflows have additional
requirements. This notebook does not attempt a generic downgrade.
[Compatibility](https://docs.delta.io/versioning/), [feature removal](https://docs.delta.io/delta-drop-feature/).

In [ ]:
for table in ['orderdb.orders', 'orderdb.column_mapping', 'orderdb.default_values', 'orderdb.liquid_clustering', 'orderdb.deletion_vectors', 'orderdb.row_tracking', 'orderdb.type_widening']:
    detail = DeltaTable.forName(spark, table).detail()
    print('PROTOCOL:', table)
    (
        detail
        .select(*[c for c in ['minReaderVersion', 'minWriterVersion', 'tableFeatures'] if c in detail.columns])
        .show(truncate=False)
    )
spark.sql("""CREATE TABLE orderdb.append_only (
    id INT, payload STRING
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/append_only'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false',
    'delta.appendOnly'='true'
)""")
spark.sql("INSERT INTO orderdb.append_only VALUES (1,'immutable logical event')").show(truncate=False)

# Optional negative example: uncomment to see the expected error.
# spark.sql('DELETE FROM orderdb.append_only WHERE id=1').collect()

## 32. Shallow clone versus an independent data copy

OSS Delta supports shallow clone. It has its own log/history but references source data files;
source VACUUM can break it. A CTAS below writes independent data files, but does not inherit
the source's complete history, constraints, and features. Do not describe either as a full
historical backup or assume Databricks deep-clone semantics.

In [ ]:
(
    spark
    .sql(f"CREATE TABLE orderdb.orders_shallow_clone SHALLOW CLONE orderdb.orders VERSION AS OF {merged_version} LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/orders_shallow_clone'")
    .show(truncate=False)
)
spark.sql('UPDATE orderdb.orders_shallow_clone SET amount=999.00 WHERE order_id=1').show(truncate=False)
spark.sql('DESCRIBE HISTORY orderdb.orders_shallow_clone').show(truncate=False)
(
    spark
    .sql(f"CREATE TABLE orderdb.orders_data_copy USING DELTA LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/orders_data_copy' AS SELECT * FROM orderdb.orders VERSION AS OF {merged_version}")
    .show(truncate=False)
)
spark.sql('SELECT * FROM orderdb.orders WHERE order_id=1').show()
spark.sql('SELECT * FROM orderdb.orders_shallow_clone WHERE order_id=1').show()

## 33. Convert disposable partitioned Parquet data to Delta

Conversion constructs a Delta log around existing Parquet files; it is not a data copy.
Quiesce all writers to a real source before conversion and use Delta-aware writers afterwards.
This lab creates its own source files and provides the partition schema explicitly.

In [ ]:
legacy = spark.createDataFrame([(1, 'south'), (2, 'north')], 'id int, region string')
(
    legacy
    .write
    .mode('errorifexists')
    .partitionBy('region')
    .parquet('hdfs://localhost:9000/user/hive/warehouse/orderdb/converted_parquet')
)
DeltaTable.convertToDelta(spark, 'parquet.`hdfs://localhost:9000/user/hive/warehouse/orderdb/converted_parquet`', 'region STRING')
(
    spark
    .sql("CREATE TABLE orderdb.converted_parquet USING DELTA LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/converted_parquet'")
    .show(truncate=False)
)
spark.sql('DESCRIBE DETAIL orderdb.converted_parquet').show(truncate=False)

## 34. Managed versus external registration, and ordinary views

Hive Metastore stores a catalog entry; an external Delta table can be re-registered from its
existing location. Dropping a managed table normally removes its data as well. This section
creates one disposable managed table and one ordinary SQL view; the view is not materialized.

In [ ]:
spark.sql('CREATE TABLE orderdb.managed_example (id INT) USING DELTA').show(truncate=False)
spark.sql('INSERT INTO orderdb.managed_example VALUES (1)').show(truncate=False)
spark.sql('DESCRIBE EXTENDED orderdb.managed_example').show(truncate=False)
(
    spark
    .sql('CREATE VIEW orderdb.high_value_orders AS SELECT order_id, amount FROM orderdb.orders WHERE amount >= 100')
    .show(truncate=False)
)
spark.sql('DROP TABLE orderdb.orders_data_copy').show(truncate=False)
(
    spark
    .sql("CREATE TABLE orderdb.orders_data_copy USING DELTA LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/orders_data_copy'")
    .show(truncate=False)
)

## 35. Identity columns in OSS Delta 3.3

Delta 3.3 adds identity-column support. Generate a surrogate value when an insert omits it,
but do not assume values are gap-free or equal to business keys. Identity columns impose
concurrent-write limitations; use a separate single-writer ingestion design for this table.
The business-key duplicate check remains your responsibility.
[3.3 release features](https://github.com/delta-io/delta/releases/tag/v3.3.0).

In [ ]:
from delta.tables import IdentityGenerator
(
    DeltaTable
    .create(spark)
    .tableName('orderdb.identity_columns')
    .location('hdfs://localhost:9000/user/hive/warehouse/orderdb/identity_columns')
    .addColumn('surrogate_id', 'BIGINT', generatedAlwaysAs=IdentityGenerator(start=1, step=1))
    .addColumn('business_key', 'STRING')
    .addColumn('label', 'STRING')
    .execute()
)
(
    spark
    .sql("INSERT INTO orderdb.identity_columns (business_key,label) VALUES ('C001','Alice'),('C002','Bob')")
    .show(truncate=False)
)
(
    spark
    .sql("INSERT INTO orderdb.identity_columns (business_key,label) VALUES ('C003','Carol')")
    .show(truncate=False)
)
spark.sql('SELECT * FROM orderdb.identity_columns ORDER BY business_key').show(truncate=False)

## 36. In-commit timestamps, checkpoints and TIMESTAMP_NTZ

In-commit timestamps store commit time in the log, improving timestamp stability when table
files move. They are a table feature, distinct from the timestamp values in your rows.
This separate table also requests frequent classic checkpoints for inspection. `TIMESTAMP_NTZ`
represents wall-clock time without a timezone; it is not a substitute for a UTC event instant.

In [ ]:
spark.sql("""
CREATE TABLE orderdb.checkpoint_lab (id INT, local_time TIMESTAMP_NTZ)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/checkpoint_lab'
TBLPROPERTIES ('delta.enableInCommitTimestamps'='true', 'delta.checkpointInterval'='2')
""")
spark.sql("INSERT INTO orderdb.checkpoint_lab VALUES (1, CAST('2026-01-01 09:00:00' AS TIMESTAMP_NTZ))")
spark.sql("INSERT INTO orderdb.checkpoint_lab VALUES (2, CAST('2026-01-01 09:00:00' AS TIMESTAMP_NTZ))")
spark.sql("INSERT INTO orderdb.checkpoint_lab VALUES (3, CAST('2026-01-01 09:00:00' AS TIMESTAMP_NTZ))")
spark.read.text(
    "hdfs://localhost:9000/user/hive/warehouse/orderdb/checkpoint_lab/_delta_log/_last_checkpoint"
).show(truncate=False)
spark.sql("DESCRIBE DETAIL orderdb.checkpoint_lab").show(truncate=False)

### Optional v2 checkpoint policy

On another disposable table, `TBLPROPERTIES ('delta.checkpointPolicy'='v2')` selects v2
checkpoints and their protocol requirements. They can include sidecar files, so operational
tooling must not assume every checkpoint is a single classic Parquet file. This changes log
representation, not your application's business checkpoint or table commit numbering.
Inspect compatibility before enabling it for shared readers.
[Table properties](https://docs.delta.io/table-properties/).

## 37. Optional Structured Streaming: bounded ingestion and restart

This uses finite JSON input files in the fixed HDFS input directory, avoiding Kafka or cloud services.
The Delta sink and checkpoint are separate directories. `availableNow` finishes after available
input is processed. Running the same query/checkpoint again must not duplicate previously
committed input. Keep checkpoints for recovery; deleting one changes replay behavior.
[Streaming guide](https://docs.delta.io/delta-streaming/).

In [ ]:
RUN_STREAMING = False
if RUN_STREAMING:
    spark.sql("""CREATE TABLE orderdb.stream_ingest (
    event_id LONG, payload STRING
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/stream_ingest'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
    first_input = spark.createDataFrame([(1, 'a'), (2, 'b'), (3, 'c')], 'event_id long, payload string')
    (
        first_input
        .coalesce(1)
        .write
        .mode('errorifexists')
        .json('hdfs://localhost:9000/user/hive/warehouse/orderdb/stream_input')
    )

    def ingest_available_files():
        source = (
            spark
            .readStream
            .schema(first_input.schema)
            .option('maxFilesPerTrigger', 2)
            .json('hdfs://localhost:9000/user/hive/warehouse/orderdb/stream_input')
        )
        query = (
            source
            .writeStream
            .format('delta')
            .outputMode('append')
            .option('checkpointLocation', 'hdfs://localhost:9000/user/hive/warehouse/orderdb/checkpoints/file_to_delta')
            .trigger(availableNow=True)
            .start('hdfs://localhost:9000/user/hive/warehouse/orderdb/stream_ingest')
        )
        try:
            if not query.awaitTermination(60):
                raise TimeoutError('Stream did not complete within 60 seconds; inspect Spark UI.')
            print(query.lastProgress)
        finally:
            query.stop()
    ingest_available_files()
    spark.sql('SELECT COUNT(*) FROM orderdb.stream_ingest').show()
    ingest_available_files()
    spark.sql('SELECT COUNT(*) FROM orderdb.stream_ingest').show()
    (
        spark
        .createDataFrame([(4, 'd'), (5, 'e')], first_input.schema)
        .coalesce(1)
        .write
        .mode('append')
        .json('hdfs://localhost:9000/user/hive/warehouse/orderdb/stream_input')
    )
    ingest_available_files()
    spark.sql('SELECT COUNT(*) FROM orderdb.stream_ingest').show()
else:
    print('Streaming skipped; set RUN_STREAMING=True to execute sections 37 and 38.')

## 38. Optional append-stream reads from Delta

Use a separate checkpoint for the Delta-to-Delta stream. An ordinary append stream is not a
complete update/delete feed. Settings such as `skipChangeCommits` deliberately omit changing
commits; they are inappropriate when the downstream table must reflect all source mutations.

In [ ]:
if RUN_STREAMING:
    spark.sql("""CREATE TABLE orderdb.stream_mirror (
    event_id LONG, payload STRING
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/stream_mirror'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
    query = (
        spark
        .readStream
        .format('delta')
        .option('maxFilesPerTrigger', 2)
        .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/stream_ingest')
        .writeStream
        .format('delta')
        .outputMode('append')
        .option('checkpointLocation', 'hdfs://localhost:9000/user/hive/warehouse/orderdb/checkpoints/delta_to_delta')
        .trigger(availableNow=True)
        .start('hdfs://localhost:9000/user/hive/warehouse/orderdb/stream_mirror')
    )
    try:
        if not query.awaitTermination(60):
            raise TimeoutError('Delta stream read did not complete within 60 seconds.')
        spark.sql('SELECT * FROM orderdb.stream_mirror').show()
    finally:
        query.stop()
else:
    print('Delta stream read skipped.')

## 39. CDF upsert consumer with tombstones and replay checks

Persist each key's source version and a tombstone for deletes. This prevents an older replay
from resurrecting a deleted key. The consumer removes update preimages and accepts one final
action per key per source version. Ambiguous same-version actions fail explicitly.
The target is seeded from `seeded_version`, then applies the captured CDF range twice to verify replay.
This design assumes a single consumer writing this target and a stable source table identity.

In [ ]:
spark.sql("""CREATE TABLE orderdb.cdf_current_state (
    order_id INT, customer_id INT, product_id INT, amount DECIMAL(12,2), source_version LONG, is_deleted BOOLEAN
)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/cdf_current_state'
TBLPROPERTIES (
    'delta.enableDeletionVectors'='false'
)""")
(
    spark
    .read
    .format('delta')
    .option('versionAsOf', seeded_version)
    .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
    .withColumn('source_version', F.lit(seeded_version).cast('long'))
    .withColumn('is_deleted', F.lit(False))
    .write
    .format('delta')
    .mode('append')
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/cdf_current_state')
)

def apply_cdf_batch(batch_df, batch_id):
    actions = batch_df.where("_change_type <> 'update_preimage'").persist()
    try:
        if actions.isEmpty():
            return
        if actions.groupBy('order_id', '_commit_version').count().where('count > 1').limit(1).count():
            raise ValueError('Multiple final actions for one key/version need an explicit source ordering policy.')
        final_actions = (
            actions
            .withColumn('rn', F.row_number().over(Window.partitionBy('order_id').orderBy(F.desc('_commit_version'))))
            .where('rn=1')
            .select('order_id', 'customer_id', 'product_id', 'amount', F.col('_commit_version').alias('source_version'), (F.col('_change_type') == 'delete').alias('is_deleted'))
        )
        (
            DeltaTable
            .forName(spark, 'orderdb.cdf_current_state')
            .alias('t')
            .merge(final_actions.alias('s'), 't.order_id=s.order_id')
            .whenMatchedUpdateAll(condition='s.source_version > t.source_version')
            .whenNotMatchedInsertAll()
            .execute()
        )
        print('Applied CDF batch', batch_id)
    finally:
        actions.unpersist()
apply_cdf_batch(cdf, 0)
apply_cdf_batch(cdf, 0)
visible_orders = (
    spark
    .table('orderdb.cdf_current_state')
    .where('NOT is_deleted')
    .select('order_id', 'customer_id', 'product_id', 'amount')
)
spark.sql('SELECT * FROM orderdb.cdf_current_state ORDER BY order_id').show(truncate=False)
visible_orders.orderBy("order_id").show()

## 40. Optional streaming CDF consumer

The same `foreachBatch` function can consume CDF. This stream deliberately starts at the
already-applied version range to exercise replay protection, then finishes with `availableNow`.
Persist an initial snapshot/version boundary before starting a real continuous consumer.
RESTORE, expired history, source replacement and schema changes all require consumer planning.

In [ ]:
RUN_CDF_STREAM = False
if RUN_CDF_STREAM:
    changes = (
        spark
        .readStream
        .format('delta')
        .option('readChangeFeed', 'true')
        .option('startingVersion', seeded_version + 1)
        .load('hdfs://localhost:9000/user/hive/warehouse/orderdb/orders')
    )
    query = (
        changes
        .writeStream
        .foreachBatch(apply_cdf_batch)
        .option('checkpointLocation', 'hdfs://localhost:9000/user/hive/warehouse/orderdb/checkpoints/cdf_consumer')
        .trigger(availableNow=True)
        .start()
    )
    try:
        if not query.awaitTermination(60):
            raise TimeoutError('CDF stream did not complete within 60 seconds.')
    finally:
        query.stop()
else:
    print('Streaming CDF skipped; batch replay checks above still run.')

## 41. Retention, VACUUM dry run and optional deletion

VACUUM deletes unreferenced files old enough for the retention rule; it does not roll back
the current table. Time travel and RESTORE can fail after their files are removed. Log retention
and deleted-file retention are separate policies. The original notebook disabled the duration
check and vacuumed at one hour; this version retains the check and previews **14 days**.
The target is the independent layout table, avoiding the orders shallow-clone dependency.

In [ ]:
RUN_VACUUM_DELETE = False
spark.conf.set('spark.databricks.delta.retentionDurationCheck.enabled', 'true')
(
    spark
    .sql("ALTER TABLE orderdb.layout_lab SET TBLPROPERTIES ('delta.deletedFileRetentionDuration'='interval 14 days', 'delta.logRetentionDuration'='interval 30 days')")
    .show(truncate=False)
)
spark.sql('VACUUM orderdb.layout_lab RETAIN 336 HOURS DRY RUN').show(truncate=False)
if RUN_VACUUM_DELETE:
    spark.sql('VACUUM orderdb.layout_lab RETAIN 336 HOURS').show(truncate=False)
else:
    print('Physical deletion skipped; only the dry run was executed.')

### VACUUM LITE and cleanup planning

Delta 3.3 adds `VACUUM ... LITE`, which uses retained log information instead of a full directory
listing. It cannot identify every arbitrary file never recorded in the log. It also has log-history
prerequisites; if the required history is unavailable it can fail with `DELTA_CANNOT_VACUUM_LITE`.
Use FULL/default VACUUM as appropriate after reviewing the retention policy.

```python
# Optional, after verifying the table has the required retained log coverage:
# q(f'VACUUM orderdb.layout_lab LITE RETAIN 336 HOURS DRY RUN')
```

Set retention longer than the longest running transaction, consumer outage/replay window, and
required historical analysis window. Schedule maintenance; a retention property alone does not
run VACUUM. Never manually delete `_delta_log`, checkpoint files, active data files, or DV files.
An OPTIMIZE rewrite and a VACUUM delete solve different problems.
[Utility commands](https://docs.delta.io/delta-utility/).

## 42. Concurrency and commit metadata

Delta uses optimistic concurrency: writers validate against intervening commits before publishing.
Conflicts can require re-reading and re-planning. Two separate table writes are not one atomic
transaction. Deletion vectors alone do not imply Databricks-style row-level concurrency here.
Attach a run label to a commit to make history easier to audit; it does not deduplicate a write.
[Concurrency reference](https://docs.delta.io/concurrency-control/).

In [ ]:
spark.sql("""
CREATE TABLE orderdb.audit_metadata (id INT, payload STRING)
USING DELTA
LOCATION 'hdfs://localhost:9000/user/hive/warehouse/orderdb/audit_metadata'
""")
audit_batch = spark.createDataFrame([(1, 'reviewed')], 'id int, payload string')
(
    audit_batch.write.format('delta').mode('append')
    .option('userMetadata', 'orders training batch 1')
    .save('hdfs://localhost:9000/user/hive/warehouse/orderdb/audit_metadata')
)
DeltaTable.forName(spark, 'orderdb.audit_metadata').history().select(
    'version', 'operation', 'userMetadata', 'operationMetrics'
).show(truncate=False)

### Two-session conflict exercise

Open a second Python kernel with the same HDFS/Hive settings and reuse the **`orderdb` database**.
Against a dedicated disposable table, issue overlapping UPDATE or MERGE jobs from both sessions.
Tiny jobs may finish before they overlap; use a larger teaching dataset to reproduce a conflict.
Record the exception and intervening history, reload the winner's state, and re-plan the loser.
Use explicit disjoint target predicates where the business partitioning permits them.
Do not blindly retry increments or operations with external side effects.

## 43. Hive/HDFS operations and recovery

| Question | On-premises approach |
|---|---|
| Where is the table? | `DESCRIBE DETAIL` location, Hive registration, `_delta_log` in HDFS |
| Is the catalog the transaction log? | No; Hive holds registration, Delta logs define table state |
| Can plain Hive read USING DELTA automatically? | No; use a compatible Delta integration/reader |
| Lost external registration? | Recreate `USING DELTA LOCATION` for the intact path, as in section 34 |
| Lost data files? | Restore from a consistent backup; a catalog repair cannot recreate them |
| Need a historical backup? | Preserve a consistent log plus every referenced file for the desired versions |
| Is shallow clone a backup? | No; it depends on source file retention |
| HDFS access control? | HDFS ownership/ACLs, authentication, and your cluster's service policies |
| Kerberos? | Configure Hadoop XML and appropriate ticket/keytab lifecycle outside notebook source |
| Checkpoint location? | Durable shared HDFS path, separate for each streaming query |

Read-only Linux shell equivalents (replace with the paths printed by this run):

```bash
hdfs dfs -ls hdfs://your-namenode:9000/user/hive/warehouse
hdfs dfs -ls hdfs://your-namenode:9000/path/to/table/_delta_log
hdfs dfs -cat hdfs://your-namenode:9000/path/to/table/_delta_log/00000000000000000001.json
```

## 44. Iceberg concepts mapped to Delta

| Iceberg workbook concept | Delta 3.3.2 equivalent / distinction |
|---|---|
| Snapshot ID | Sequential table commit version |
| Snapshots/history metadata tables | DESCRIBE HISTORY, DESCRIBE DETAIL, DeltaTable API, `_delta_log` |
| VERSION/TIMESTAMP AS OF | Same query concept; use Delta version IDs and retained log/files |
| Rollback pointer | RESTORE writes a new commit |
| Snapshot expiration + orphan cleanup | Log cleanup and VACUUM have different retention responsibilities |
| Branches, tags, fast-forward, cherry-pick | No native Iceberg-style equivalents in this OSS setup |
| Audit branch publication | Stage a separate table, validate, then MERGE or deliberately replace |
| Hidden partition transforms/spec evolution | Explicit partitions or generated columns; rewrite for partition changes |
| COW / MOR position-delete files | COW or deletion vectors; representations differ |
| Incremental append scan | Delta append stream; CDF for logical changes |
| Field-ID evolution | Enable column mapping for metadata-only rename/drop |
| Identifier fields | Business-key validation; identity columns generate values but do not validate source keys |
| SparkCatalog procedures | Delta SQL commands and Python DeltaTable methods |

## 45. Coverage and OSS/Databricks boundaries

| Feature | This workbook / scope |
|---|---|
| Hive, HDFS, Python/SQL, named/path tables | On-premises setup throughout |
| CRUD, SQL/Python MERGE, source deduplication | 6–14 |
| Time travel, RESTORE, diffs | 10–12 |
| CDF, idempotent transactions, SCD2, replay-safe target | 15–17, 39–40 |
| Additive/replacement schema, nested fields, column mapping | 18–20 |
| Constraints, generated/default/identity columns | 21–22, 35 |
| Partitioning, replaceWhere, dynamic overwrite | 23–24 |
| OPTIMIZE, Z-order, optimized writes, auto compaction | 25–26 |
| Liquid clustering / OPTIMIZE FULL | 27; OSS, not only Databricks |
| Deletion vectors / REORG, row tracking, type widening | 28–30 |
| Protocol/feature inspection, append-only | 31 |
| Shallow clone, independent copy, Parquet conversion | 32–34 |
| In-commit timestamps, checkpoints, TIMESTAMP_NTZ | 36 |
| Streaming source/sink, CDF foreachBatch | 37–40, opt-in |
| VACUUM/default and LITE recipe, retention | 41 |
| Concurrency, operation metrics and commit labels | 42 |
| UniForm / Iceberg metadata conversion | Additional matched artifacts and compatibility requirements; recipe below |
| Symlink manifests | Optional interoperability recipe below; not a general modern-feature reader |
| Delta Sharing, Flink, Trino, Kernel | Separate clients/services; not provided by the two Spark JARs alone |
| Unity Catalog/DBFS/dbutils/Auto Loader/Photon | Databricks-specific source assumptions removed; use on-prem alternatives |
| Deep clone, predictive optimization, managed pipelines | Do not assume Databricks commands exist in OSS 3.3.2 |
| Primary/foreign-key enforcement, multi-table atomicity | Not supplied by these examples |
| Iceberg-like branches/tags | Not native to this setup; shallow clone has different semantics |
| Newer 4.x types/features and catalog-managed commits | Outside the pinned 3.3.2 curriculum |

**Interoperability recipes, not Run All steps:**

```python
# For a compatible consumer and a table without unsupported newer features:
# DeltaTable.forName(spark, 'orderdb.orders_data_copy').generate('symlink_format_manifest')
```

UniForm needs the matching `io.delta:delta-iceberg_2.12:3.3.2` integration artifact in addition
to the core JARs and a compatible table feature set. Configure it in a fresh session and a
separate table, then validate the external Iceberg reader:

```sql
-- Recipe only: configure the integration first, and inspect all compatibility requirements.
ALTER TABLE db.uniform_candidate SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.enableIcebergCompatV2' = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
);
```

Generated Iceberg metadata is a compatibility view of a Delta-owned table; do not introduce
independent Iceberg writes to those shared files. [UniForm](https://docs.delta.io/delta-uniform/).

## 46. Troubleshooting

| Symptom | Check |
|---|---|
| `No module named delta` | Install the Delta Python wheel in this kernel; JARs alone are insufficient |
| `JavaPackage is not callable` / class not found | Delta JAR missing from driver/executor classpath |
| `NoSuchMethodError` / Scala linkage | Spark minor, Scala binary, duplicate Delta JARs, wheel/JVM mismatch |
| DELTA_CONFIGURE_SPARK_SESSION_WITH_EXTENSION_AND_CATALOG | Extension and DeltaCatalog set before SparkContext; restart kernel |
| Metastore connection refused | Thrift endpoint, host resolution, service, network access |
| Executor HDFS failure | NameNode hostname, Hadoop XML, Kerberos/ACLs, driver/executor connectivity |
| SQL magic unknown | Execute section 4 in a Python kernel |
| Syntax error from `%python` or raw SQL | Those source cells were Databricks cells; use this workbook's Python/%%sql format |
| Schema mismatch | Match the source types or explicitly review and enable schema evolution |
| Constraint error | Correct invalid rows; the rejected batch should not have committed |
| Concurrent modification | Inspect history and re-plan a replay-safe operation |
| Time-travel/RESTORE missing files | Retention or file loss; log history alone cannot restore removed Parquet |
| CDF unavailable at starting version | Enable before the required changes; check retention and schema boundaries |
| Table already exists on rerun | Inspect existing tables; reset only your training tables/data before repeating |
| Streaming stops after schema change | Coordinate schema/consumer upgrades and checkpoint/schema tracking behavior |
| Preview/feature error on another runtime | Recheck exact version and reader/writer protocol support |

## 47. Exercises and expected outcomes

1. After DELETE, current orders are empty, but `insert_version` has order 1 at 123.40.
2. RESTORE has a version greater than the restored historical version.
3. After MERGE, IDs are 1, 3, 6, 7, 8, with order 1 at 200.00.
4. CDF contains a delete for order 4 and update images for order 1.
5. Replaying the append transaction does not add duplicate rows.
6. Replaying SCD2 leaves four historical rows and three current rows.
7. A column-mapping rename/drop keeps the same active file set.
8. A negative amount fails the CHECK and does not create a committed version.
9. Partition replacement preserves the other date's records.
10. Compaction and REORG preserve the logical contents.
11. A shallow-clone update leaves the source order amount unchanged.
12. Replaying the CDF consumer preserves its tombstone and final visible rows.
13. If enabled, streaming restarts with the same checkpoint keep five total input events.

## 48. Review the final tables

The main orders table should contain IDs 1, 3, 6, 7, and 8. The SCD table should have four
historical rows and three current rows. Inspect the results directly below.

In [ ]:
spark.sql("SHOW TABLES IN orderdb").show(truncate=False)
spark.sql("SELECT * FROM orderdb.orders ORDER BY order_id").show()
spark.sql("SELECT * FROM orderdb.customer_scd2 ORDER BY customer_id, effective_from").show()
spark.sql("SELECT * FROM orderdb.cdf_current_state ORDER BY order_id").show()

## 49. Optional cleanup

Uncomment only the commands for training tables you want to remove. These external table
drops remove registrations but retain data. Re-register a retained table with `USING DELTA
LOCATION` to inspect it. To repeat from empty tables, archive or remove the old training data
separately before rerunning CREATE. The managed example's DROP can delete its data.
The database, shared clone/source files, and streaming checkpoints are not automatically removed.

In [ ]:
# spark.sql("DROP VIEW IF EXISTS orderdb.high_value_orders")
# spark.sql("DROP TABLE IF EXISTS orderdb.orders")
# spark.sql("DROP TABLE IF EXISTS orderdb.python_api")
# spark.sql("DROP TABLE IF EXISTS orderdb.idempotent_append")
# spark.sql("DROP TABLE IF EXISTS orderdb.customer_scd2")
# spark.sql("DROP TABLE IF EXISTS orderdb.schema_evolution")
# spark.sql("DROP TABLE IF EXISTS orderdb.merge_schema_evolution")
# spark.sql("DROP TABLE IF EXISTS orderdb.column_mapping")
# spark.sql("DROP TABLE IF EXISTS orderdb.schema_replacement")
# spark.sql("DROP TABLE IF EXISTS orderdb.constraints_lab")
# spark.sql("DROP TABLE IF EXISTS orderdb.generated_dates")
# spark.sql("DROP TABLE IF EXISTS orderdb.default_values")
# spark.sql("DROP TABLE IF EXISTS orderdb.daily_orders")
# spark.sql("DROP TABLE IF EXISTS orderdb.orders_by_region")
# spark.sql("DROP TABLE IF EXISTS orderdb.layout_lab")
# spark.sql("DROP TABLE IF EXISTS orderdb.liquid_clustering")
# spark.sql("DROP TABLE IF EXISTS orderdb.deletion_vectors")
# spark.sql("DROP TABLE IF EXISTS orderdb.row_tracking")
# spark.sql("DROP TABLE IF EXISTS orderdb.type_widening")
# spark.sql("DROP TABLE IF EXISTS orderdb.append_only")
# spark.sql("DROP TABLE IF EXISTS orderdb.orders_shallow_clone")
# spark.sql("DROP TABLE IF EXISTS orderdb.orders_data_copy")
# spark.sql("DROP TABLE IF EXISTS orderdb.converted_parquet")
# spark.sql("DROP TABLE IF EXISTS orderdb.identity_columns")
# spark.sql("DROP TABLE IF EXISTS orderdb.checkpoint_lab")
# spark.sql("DROP TABLE IF EXISTS orderdb.stream_ingest")
# spark.sql("DROP TABLE IF EXISTS orderdb.stream_mirror")
# spark.sql("DROP TABLE IF EXISTS orderdb.cdf_current_state")
# spark.sql("DROP TABLE IF EXISTS orderdb.audit_metadata")
# spark.sql("DROP TABLE IF EXISTS orderdb.managed_example")
# spark.stop()

## References and validation status

The notebook's code targets **Delta 3.3.2 / Spark 3.5 / Scala 2.12**. Public documentation may
describe newer features as well; follow the version labels and the
[3.3 release](https://github.com/delta-io/delta/releases/tag/v3.3.0) /
[3.3.2 release](https://github.com/delta-io/delta/releases/tag/v3.3.2).
The [3.3.2 Python API source](https://github.com/delta-io/delta/blob/v3.3.2/python/delta/tables.py)
is a pinned reference for DeltaTable methods.

Further references: [batch](https://docs.delta.io/delta-batch/),
[DML](https://docs.delta.io/delta-update/), [CDF](https://docs.delta.io/delta-change-data-feed/),
[streaming](https://docs.delta.io/delta-streaming/),
[operations](https://docs.delta.io/delta-utility/),
[table properties](https://docs.delta.io/table-properties/),
[storage](https://docs.delta.io/delta-storage/).

**Authoring validation:** notebook schema and transformed Python cell syntax are checked by the
companion builder; the SQL magic sends its cell directly to Spark. Outputs are deliberately
empty. The authoring workspace has no running Spark/HDFS/Hive lab, so this file is **not presented
as end-to-end executed**. Compare the displayed results with the expected outcomes when running on your cluster.